In [1]:
from pyspark.sql.session import SparkSession

spark = (

    SparkSession
    .builder
    .appName("spark-sql-aware")
    .config("spark.sql.warehouse.dir","../data/warehouse")
    .getOrCreate()
)

spark

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
/home/suanki/Desktop/pyspark-env/lib/python3.13/site-packages/pyspark/testing/utils.py:127: FutureWarning: PySpark does not yet fully support pandas >= 3.0.0. Some features may not work correctly. It is recommended to use pandas < 3.0.0 for now.
  require_minimum_pandas_version()
26/11/13 14:06:03 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [3]:
import os

rel = "../data/optimization-data/customer"
abs_path = os.path.abspath(rel)

spark.sql(f"create table if not exists customer using parquet options(path '{abs_path}')")

DataFrame[]

In [5]:
spark.sql("select * from customer").show(10,truncate=False)

+----------+-------------+---+------+----------+-----+------------+
|cust_id   |name         |age|gender|birthday  |zip  |city        |
+----------+-------------+---+------+----------+-----+------------+
|C007YEYTX9|Aaron Abbott |34 |Female|7/13/1991 |97823|boston      |
|C00B971T1J|Aaron Austin |37 |Female|12/16/2004|30332|chicago     |
|C00WRSJF1Q|Aaron Barnes |29 |Female|3/11/1977 |23451|denver      |
|C01AZWQMF3|Aaron Barrett|31 |Male  |7/9/1998  |46613|los_angeles |
|C01BKUFRHA|Aaron Becker |54 |Male  |11/24/1979|40284|san_diego   |
|C01RGUNJV9|Aaron Bell   |24 |Female|8/16/1968 |86331|denver      |
|C01USDV4EE|Aaron Blair  |35 |Female|9/9/1974  |80078|new_york    |
|C01WMZQ7PN|Aaron Brady  |51 |Female|8/20/1994 |52204|philadelphia|
|C021567NJZ|Aaron Briggs |57 |Male  |3/10/1990 |22008|philadelphia|
|C023M6MKR3|Aaron Bryan  |29 |Male  |4/10/1976 |05915|philadelphia|
+----------+-------------+---+------+----------+-----+------------+
only showing top 10 rows


In [19]:

spark.sql("""
    create or replace view   customer_view 
          as
          select cust_id, (split(name,' ')[0] as firstname,split(name,' ')[1] as lastname) as name  from customer where city='boston'
""")

DataFrame[]

In [21]:
spark.sql("select * from customer_view").printSchema()

root
 |-- cust_id: string (nullable = true)
 |-- name: struct (nullable = false)
 |    |-- firstname: string (nullable = true)
 |    |-- lastname: string (nullable = true)



In [22]:
spark.sql("select name.firstname from customer_view").show(5, False)

+---------+
|firstname|
+---------+
|Aaron    |
|Aaron    |
|Aaron    |
|Aaron    |
|Aaron    |
+---------+
only showing top 5 rows


In [23]:
spark.sql("select * from customer_view").show(5, False)

+----------+-----------------+
|cust_id   |name             |
+----------+-----------------+
|C007YEYTX9|{Aaron, Abbott}  |
|C08XAQUY73|{Aaron, Lambert} |
|C094P1VXF9|{Aaron, Lindsey} |
|C097SHE1EF|{Aaron, Lopez}   |
|C0DTC6436T|{Aaron, Schwartz}|
+----------+-----------------+
only showing top 5 rows


In [32]:
spark.sql(""" 

select  city, collect_list(name) as people_list, size(collect_list(name)) as total_people  from customer 
          group by city,name
          having size(collect_list(name) ) > 1

""").show(truncate=False)

+------------+------------------------------------+------------+
|city        |people_list                         |total_people|
+------------+------------------------------------+------------+
|boston      |[Brandon Hoffman, Brandon Hoffman]  |2           |
|boston      |[Cameron Duncan, Cameron Duncan]    |2           |
|chicago     |[Alex Holloway, Alex Holloway]      |2           |
|chicago     |[Billy Watts, Billy Watts]          |2           |
|denver      |[Brent Conner, Brent Conner]        |2           |
|los_angeles |[Adeline Gibson, Adeline Gibson]    |2           |
|los_angeles |[Bettie Gonzalez, Bettie Gonzalez]  |2           |
|los_angeles |[Clara Figueroa, Clara Figueroa]    |2           |
|new_york    |[Ada Benson, Ada Benson]            |2           |
|new_york    |[Alta Christensen, Alta Christensen]|2           |
|new_york    |[Alta McBride, Alta McBride]        |2           |
|new_york    |[Carolyn Payne, Carolyn Payne]      |2           |
|philadelphia|[Brandon Wa

### Functions

In [34]:
spark.sql("show functions").show()

+--------+
|function|
+--------+
|       !|
|      !=|
|       %|
|       &|
|       *|
|       +|
|       -|
|       /|
|       <|
|      <<|
|      <=|
|     <=>|
|      <>|
|       =|
|      ==|
|       >|
|      >=|
|      >>|
|     >>>|
|       ^|
+--------+
only showing top 20 rows
